In [1]:
from functools import reduce
from pyspark.sql.functions import lit, col, current_timestamp

dbutils.widgets.text("catalog", "transport_vic_dev")
catalog = dbutils.widgets.get("catalog")

folders = ["1", "2", "3", "4", "5", "6", "10", "11"]

def read_stops(folder):
    return (spark.read
            .option("header", True)
            .option("inferSchema", False)          # bronze = raw strings; cast in silver
            .csv(f"/Volumes/{catalog}/01_bronze/gtfs_data/{folder}/google_transit/stops.txt")
            .withColumn("feed_id", lit(folder))            # (feed_id, stop_id) is the real key
            .withColumn("_source_file", col("_metadata.file_path"))  # UC-supported (input_file_name() is not)
            .withColumn("_ingest_ts", current_timestamp()))

stops = reduce(lambda a, b: a.unionByName(b, allowMissingColumns=True),
               [read_stops(f) for f in folders])

tbl = f"{catalog}.`01_bronze`.stops"
stops.write.mode("overwrite").saveAsTable(tbl)

/Users/jacktoke/Codes/Databricks/transport-vic-project/.venv/lib/python3.12/site-packages/databricks/sdk/_widgets/__init__.py:70: UserWarning: 
To use databricks widgets interactively in your notebook, please install databricks sdk using:
	pip install 'databricks-sdk[notebook]'
Falling back to default_value_only implementation for databricks widgets.
  warnings.warn(
